# Multimodal Topic Mining and Image Classification
## Advanced Customer Analytics - Visual Data Predictions Assignment


In [ ]:
# Install required packages
# !pip install -q bertopic
# !pip install -q scikit-learn
# !pip install -q pillow
# !pip install -q pandas
# !pip install -q matplotlib
# !pip install -q seaborn
# !pip install -q requests
# !pip install -q tqdm
# !pip install -q datasets

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset

# BERTopic
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import VisualRepresentation
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# ML models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def download_from_huggingface(num_cards=1000, save_dir='pokemon_cards'):
    """
    Download Pokemon card dataset from Hugging Face repository.

    Args:
        num_cards: Number of cards to download
        save_dir: Directory to save images and metadata

    Returns:
        image_paths: List of paths to saved images
        captions: List of card descriptions
        cards_data: List of card metadata dictionaries
    """
    from datasets import load_dataset

    os.makedirs(f"{save_dir}/images", exist_ok=True)

    print("Loading dataset from Hugging Face...")
    dataset = load_dataset("TheFusion21/PokemonCards", split="train")
    dataset = dataset.select(range(min(num_cards, len(dataset))))

    image_paths = []
    captions = []
    cards_data = []

    print(f"Processing {len(dataset)} cards...")

    for idx, item in enumerate(tqdm(dataset, desc="Downloading cards")):
        try:
            img_url = item['image_url']

            if not img_url:
                continue

            response = requests.get(img_url, timeout=10)
            response.raise_for_status()

            img = Image.open(BytesIO(response.content)).convert('RGB')
            img_path = f"{save_dir}/images/card_{idx:05d}.jpg"
            img.save(img_path, 'JPEG', quality=95)

            caption = item.get('caption', item.get('name', 'Pokemon card'))

            image_paths.append(img_path)
            captions.append(caption)
            cards_data.append({
                'id': item.get('id', f'card_{idx}'),
                'name': item.get('name', 'Unknown'),
                'hp': item.get('hp', 'Unknown'),
                'set_name': item.get('set_name', 'Unknown'),
                'image_path': img_path,
                'caption': caption
            })

        except Exception as e:
            if idx < 5:
                print(f"\nError processing card {idx}: {e}")
            continue

    df = pd.DataFrame(cards_data)
    df.to_csv(f'{save_dir}/pokemon_cards_metadata.csv', index=False)

    print(f"\nSuccessfully downloaded {len(image_paths)} cards.")
    return image_paths, captions, cards_data

In [ ]:
print("Initiating dataset download...")
print()

image_paths, captions, cards_metadata = download_from_huggingface(num_cards=1000)

print("\n" + "=" * 70)
print("SAMPLE CAPTIONS")
print("=" * 70)
for i in range(min(5, len(captions))):
    print(f"\n{i+1}. {captions[i][:200]}...")